# [7.3] Mini Activation Oracles - Solutions

This notebook runs the reference implementations, local tests, notebook contract, and report-backed signature result.

<details>
<summary>Help - what to compare with your exercise notebook</summary>

The reference solution builds the core mini-oracle loop: question-conditioned rows, trained question-conditioned MLP, independently trained activation-only baselines, OOD split reports, random abstention, and clean-to-corrupt substitution.

</details>

<details>
<summary>Expected output</summary>

All local tests should pass, and the committed CUDA report should show the pinned `gelu-1l` preflight passing.

</details>


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt

chapter = "chapter7_activation_to_language"
section = "part3_mini_activation_oracles"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part3_mini_activation_oracles.solutions as solutions
import part3_mini_activation_oracles.tests as tests


## Local Tests

<details>
<summary>Expected output</summary>

Each test prints `All tests ... passed!`, including `test_notebook_contract`.

</details>

<details>
<summary>Help - why these tests are useful</summary>

They protect the moves that distinguish an oracle from a probe: question ids, independently trained baselines, separate OOD splits, abstention, and answer-changing substitution.

</details>


In [ ]:
tests.test_build_activation_question_batch_validates_shapes_and_questions(
    solutions.build_activation_question_batch,
    solutions.default_activation_questions,
)
tests.test_build_activation_question_batch_rejects_out_of_range_question_ids(
    solutions.build_activation_question_batch,
)
tests.test_question_conditioned_oracle_uses_question_ids_not_copied_probe_logits(
    solutions.make_question_conditioned_rows,
    solutions.train_question_conditioned_oracle,
    solutions.oracle_logits_for_batch,
)
tests.test_oracle_comparison_report_beats_text_and_probe_baselines(
    solutions.oracle_comparison_report,
)
tests.test_template_split_and_ood_reports_expose_generalization_failures(
    solutions.split_accuracy_by_template,
    solutions.ood_generalization_report,
)
tests.test_ood_generalization_report_rejects_invalid_threshold(
    solutions.ood_generalization_report,
)
tests.test_random_activation_report_requires_abstention_or_low_confidence(
    solutions.random_activation_oracle_report,
)
tests.test_random_activation_report_rejects_bad_rank_and_thresholds(
    solutions.random_activation_oracle_report,
)
tests.test_activation_patching_report_checks_answer_change(
    solutions.activation_patching_oracle_report,
)
tests.test_activation_patching_report_rejects_incompatible_logits(
    solutions.activation_patching_oracle_report,
)
tests.test_notebook_contract(solutions.run_smoke_test)


## Notebook Contract

<details>
<summary>Expected output</summary>

The contract should include the question bank, batch metadata, oracle-vs-baseline comparison, template split, OOD report, random-activation report, and patching answer flip.

</details>

<details>
<summary>Help - how to read it</summary>

The contract is the CPU miniature of the CUDA preflight. It proves the pieces compose before we inspect the real-model result.

</details>


In [ ]:
contract = solutions.run_smoke_test(cpu=True)
contract


## Signature Result

<details>
<summary>Expected output</summary>

`preflight_passed == True`, oracle `1.0`, text/probe baselines `0.5`, all OOD splits `1.0`, random abstention `1.0`, and patching answer `1 -> 0`.

</details>

<details>
<summary>Help - interpreting the result</summary>

The signature result is convincing because several shortcut explanations fail together: text-only cannot solve it, activation-only probes cannot solve contradictory question rows, and random activations abstain.

</details>


In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
assert gpu["preflight_passed"]
assert gpu["oracle_accuracy"] == 1.0
assert gpu["text_only_accuracy"] == 0.5
assert gpu["activation_only_probe_accuracy_max"] <= 0.75
assert gpu["question_id_changes_predictions"] == 1.0
assert gpu["passes_ood"]
assert gpu["random_graceful_failure"]
assert gpu["patching_changed_answer"]
summary = {
    "model": gpu["model_name"],
    "activation_shape": gpu["activation_shape"],
    "rows_questions": (gpu["question_conditioned_row_count"], gpu["question_count"]),
    "oracle_vs_text": (gpu["oracle_accuracy"], gpu["text_only_accuracy"]),
    "probe_baselines": (gpu["linear_probe_accuracy"], gpu["mlp_probe_accuracy"], gpu["sae_classifier_accuracy"]),
    "ood": (gpu["heldout_template_accuracy"], gpu["new_name_accuracy"], gpu["long_context_accuracy"], gpu["adversarial_accuracy"]),
    "random": (gpu["random_abstention_rate"], gpu["random_mean_confidence"]),
    "patch": (gpu["original_answer"], gpu["patched_answer"]),
    "peak_vram_gb": gpu["peak_vram_gb"],
}
summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].bar(
    ["oracle", "text", "linear", "MLP", "SAE"],
    [gpu["oracle_accuracy"], gpu["text_only_accuracy"], gpu["linear_probe_accuracy"], gpu["mlp_probe_accuracy"], gpu["sae_classifier_accuracy"]],
    color=["#0891b2", "#94a3b8", "#94a3b8", "#94a3b8", "#94a3b8"],
)
axes[0].axhline(0.75, color="#64748b", linestyle="--", linewidth=1)
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Oracle vs baselines")
axes[1].bar(
    ["heldout", "new", "long", "adv"],
    [gpu["heldout_template_accuracy"], gpu["new_name_accuracy"], gpu["long_context_accuracy"], gpu["adversarial_accuracy"]],
    color="#16a34a",
)
axes[1].set_ylim(0, 1.05)
axes[1].set_title("OOD split accuracies")
fig.tight_layout()
plt.show()


## Limitations

This validates a GT-1 local mini-oracle preflight on one pinned `gelu-1l` residual hook. It is not a LoRA-trained or API-backed Activation Oracle benchmark, not open-ended semantic QA, and not a full downstream generation patch.

## Further Research

Add more question families, multiple layers, multiple training seeds, stronger random controls, and a local LoRA oracle when the evidence contract is ready.


In [ ]:
def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu_report = report["metrics"]["gpu_test"]
    assert gpu_report["peak_vram_gb"] <= max_vram_gb
    return gpu_report


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


run_full_experiment(max_vram_gb=24.0)
